# 01 — Velas

- La **última** vela cerrada.
- Un **listado** de las N últimas.
- Un **rango de fechas** (lo que necesita un backtest).

Todo es público: funciona aunque las credenciales no estén bien.

### Dos avisos que importan

**`market` no es cosmético.** El perpetuo y el contado del mismo par son
series de precios distintas (base, funding). Pedir velas de spot para operar
futuros mueve los niveles que calcule cualquier estrategia.

**Bitunix spot no sabe paginar.** Su kline devuelve *siempre* las mismas
~201 velas más recientes e ignora `limit`, `startTime` y `endTime`. Por eso
`get_historical_candles` usa el kline de *futuros* incluso pidiendo spot —
es la única fuente de Bitunix con histórico. Está documentado en
`market_data/bitunix.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # el repo, sin instalarlo

from luarix_exchange.brokers.factory import AvailableBrokers, BrokerFactory
from luarix_exchange.brokers.credentials import BrokerCredentials, load_dotenv

load_dotenv()   # lee el .env de la raiz del repo

# Cambia esto para probar el otro broker. Todo lo demas del cuaderno
# funciona igual: esa es justamente la idea del contrato unico.
BROKER   = AvailableBrokers.BITUNIX     # o AvailableBrokers.BINANCE
MARKET   = "futures"                    # o "spot"
SYMBOL   = "BTCUSDT"

# Sin `credentials=`, la factory las lee del .env.
# En produccion el backend las pasa explicitas, una por usuario:
#     BrokerFactory.get_broker(BROKER, credentials=BrokerCredentials(
#         api_key=fila.api_key, api_secret=fila.api_secret))
broker = BrokerFactory.get_broker(
    BROKER,
    testnet=False,          # Bitunix lo ignora: no tiene testnet
    test_order=False,       # solo Binance: True valida sin ejecutar
    trading_mode="live",
    allow_live_trading=False,   # sigue bloqueando el place_order real
)
print(type(broker).__name__, "| futuros:", broker.supports_futures, "| OCO:", broker.supports_oco)

## La última vela cerrada

In [ ]:
vela = broker.get_last_candle(SYMBOL, interval="1h", market=MARKET)
vela

In [ ]:
print(f"{vela.symbol} {vela.timeframe}  {vela.open_time:%Y-%m-%d %H:%M} UTC")
print(f"  O={vela.open}  H={vela.high}  L={vela.low}  C={vela.close}  vol={vela.volume}")

## Un listado grande

Orden **ascendente** siempre: la última del listado es la más reciente. Los
dos exchanges devuelven sus klines al revés y el paquete los da la vuelta,
porque cualquier indicador da por hecho el orden ascendente.

El máximo por llamada lo pone el exchange: 1000 en Binance, ~200 en Bitunix.

In [ ]:
velas = broker.get_candles(SYMBOL, interval="1h", limit=200, market=MARKET)
print(len(velas), "velas")
print("primera:", velas[0].open_time)
print("ultima :", velas[-1].open_time)

In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "date": c.open_time, "open": c.open, "high": c.high,
    "low": c.low, "close": c.close, "volume": c.volume,
} for c in velas])
df.tail(10)

### Comprobación: ¿hay huecos?

Un hueco *en medio* invalida cualquier cálculo de indicadores. Que la serie
*termine* antes de lo pedido, en cambio, solo significa que el exchange no
tiene más — no es un error.

In [ ]:
huecos = df["date"].diff().value_counts()
print(huecos.head())
print("\nhuecos distintos del paso normal:", max(0, len(huecos) - 1))

## Un rango de fechas

Esto sí pagina contra el exchange: puede costar varias llamadas y tardar.

In [ ]:
from datetime import datetime, timedelta, timezone

fin = datetime.now(timezone.utc)
inicio = fin - timedelta(days=30)

historico = broker.get_historical_candles(
    SYMBOL, interval="4h", start=inicio, end=fin, market=MARKET,
)
print(len(historico), "velas de 4h")
print(historico[0].open_time, "->", historico[-1].open_time)

## Spot y futuros lado a lado

Deben parecerse mucho y **no** ser idénticos. Si dan exactamente lo mismo,
sospecha: significa que el selector de mercado no está llegando al endpoint.

In [ ]:
try:
    spot = broker.get_last_candle(SYMBOL, "1h", market="spot")
    fut  = broker.get_last_candle(SYMBOL, "1h", market="futures")
    print(f"spot    close={spot.close}")
    print(f"futuros close={fut.close}")
    print(f"diferencia={abs(spot.close - fut.close):.4f}")
except Exception as e:
    print(type(e).__name__, ":", e)

## El libro de órdenes

In [ ]:
libro = broker.get_order_book(SYMBOL, limit=5, market=MARKET)
libro